In [6]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer


In [7]:
# Function for preprocessing text
def preprocess_text(text):
    words = nltk.word_tokenize(text.lower())
    stop_words = set(stopwords.words('english'))  # Cache stopwords for efficiency
    filtered_words = [word for word in words if word.isalnum() and word not in stop_words]
    return ' '.join(filtered_words)

# Function for vectorizing text
def vectorize_text(processed_text, max_features=20000):
    vectorizer = TfidfVectorizer(max_features=max_features)  # Increase max features for a larger dataset
    text_vectors = vectorizer.fit_transform([processed_text]).toarray()
    return text_vectors, vectorizer


In [8]:
# Read the dataset
with open('text8', 'r') as file:
    text_data = file.read()

# Display dataset details
print(f"Dataset length: {len(text_data)} characters")
print(f"Sample text: {text_data[:500]}")

Dataset length: 100000000 characters
Sample text:  anarchism originated as a term of abuse first used against early working class radicals including the diggers of the english revolution and the sans culottes of the french revolution whilst the term is still used in a pejorative way to describe any act that used violent means to destroy the organization of society it has also been taken up as a positive label by self defined anarchists the word anarchism is derived from the greek without archons ruler chief king anarchism as a political philoso


In [13]:
# Configure chunk and batch sizes
chunk_size = 1000000  # Process 1 million characters per chunk
batch_size = 1000  # Number of chunks to process in a batch

print(f"Chunk size: {chunk_size}")
print(f"Batch size: {batch_size}")

Chunk size: 1000000
Batch size: 1000


In [14]:
# Process and vectorize in batches
all_vectors = []
total_chunks = (len(text_data) + chunk_size - 1) // chunk_size  # Calculate total chunks
range_start = 1

for i in range(0, total_chunks, batch_size):
    # Prepare batch of chunks
    batch = []
    for j in range(i, min(i + batch_size, total_chunks)):
        start_idx = j * chunk_size
        end_idx = min((j + 1) * chunk_size, len(text_data))
        chunk = text_data[start_idx:end_idx]
        processed_chunk = preprocess_text(chunk)
        batch.append(processed_chunk)
    
    # Combine and vectorize the batch
    combined_batch = ' '.join(batch)  # Combine chunks in the batch for vectorization
    batch_vectors, vectorizer = vectorize_text(combined_batch)
    all_vectors.append(batch_vectors)
    
    # Print progress
    batch_start = i + 1
    batch_end = min(i + batch_size, total_chunks)
    print(f"Processed chunks {batch_start}-{batch_end}")

print(f"Total chunks processed: {len(all_vectors)}")


Processed chunks 1-100
Total chunks processed: 1


In [11]:
# Combine all vectors if needed
final_vectors = np.vstack(all_vectors)  # Stack vectors into a single array
print(f"Final vector shape: {final_vectors.shape}")


Final vector shape: (1, 20000)


In [12]:
# Perform K-Means clustering
n_clusters = 5  # Define the number of clusters
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans.fit(final_vectors)

# Display cluster information
print("Cluster centers:")
print(kmeans.cluster_centers_)
print("Cluster labels:")
print(kmeans.labels_)


ValueError: n_samples=1 should be >= n_clusters=5.